# Multi-modal Model V8-Team - Team Only (No Position Zone)

이 노트북은 Tuned H96에 team_id만 추가한 버전입니다 (position_zone 제거).

## 주요 변경 사항 (V8-Team)

1. **Team ID Embedding** (dim=8):
   - 팀별 플레이 스타일 학습
2. **Position Zone 제거**:
   - start 좌표에서 자동 계산하는 zone은 노이즈로 판단되어 제거
   - 팀 정보만 사용하여 더 깔끔한 학습
3. **Preprocessor**: preprocessing_multimodal_team.py
4. **Model**: MultiModalNetV8Team

## 기대 효과
- Tuned H96: 13.6
- V8-Zone (with position): 13.6
- **V8-Team 목표: 13.0~13.5** (노이즈 제거로 개선 기대)


In [1]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence, pad_sequence
from sklearn.model_selection import GroupKFold
import cv2  # OpenCV Import
from tqdm import tqdm
import joblib

from src.models_multimodal_v8_team import MultiModalNetV8Team, EuclideanLoss, MaskedSeqEuclideanLoss

# [v7 Fix] Use Fixed Preprocessor (No Rounding)
from src.preprocessing_multimodal_team import FootballPreprocessorMultimodalTeam

# Fold-specific Seeds for Diversity
FOLD_SEEDS = {
    1: 42,
    2: 43,
    3: 44,
    4: 45,
    5: 46,
    6: 47,
    7: 48,
    8: 49,
    9: 50,
    10: 51,
}

# Set Seed
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

Device: cuda


## 1. 데이터 로드 및 전처리 (Preprocessing V2.5)

## 2. 이미지 생성 및 증강 (Random Y-Flip)

In [2]:
# Load Raw Training Data
train_df = pd.read_csv("train.csv")
print(f"Loaded {len(train_df)} rows")

Loaded 356721 rows


In [3]:
class MultiModalDataset(Dataset):
    def __init__(self, episodes, img_size=(68, 105), augment=False, cache_images=True):
        self.episodes = episodes
        self.H, self.W = img_size
        self.augment = augment
        self.cache_images = cache_images
        self.img_cache = None
        
        # [v7 Fix] Image Caching (Speed Optimization)
        if self.cache_images:
            print(f"Pre-rendering {len(episodes)} images (Cache Enabled)...")
            # Generate Base Images (No Augmentation applied yet)
            self.img_cache = [self._generate_image(ep['cont']) for ep in tqdm(episodes, desc="Caching Images")]

    def __len__(self):
        return len(self.episodes)

    def _generate_image(self, cont_data):
        # [v6 Optimization] Fast Generation with OpenCV
        img_np = np.zeros((2, self.H, self.W), dtype=np.float32)
        if len(cont_data) == 0:
             return torch.tensor(img_np, dtype=torch.float32)

        seq_len = cont_data.shape[0]
        
        xs = cont_data[:, 0]
        ys = cont_data[:, 1]
        
        # 좌표 변환 (0~1 -> Grid)
        # Note: Input is float (fixed preprocessor), map to int index
        x_idxs = (xs * (self.W - 1)).round().astype(int).clip(0, self.W - 1)
        y_idxs = (ys * (self.H - 1)).round().astype(int).clip(0, self.H - 1)
        
        for t in range(seq_len):
            px, py = x_idxs[t], y_idxs[t]
            
            # Channel 0: Dots
            img_np[0, py, px] = 1.0
            
            decay_val = (t + 1) / seq_len
            
            # Channel 1: Line
            if t > 0:
                prev_px, prev_py = x_idxs[t-1], y_idxs[t-1]
                cv2.line(img_np[1], (prev_px, prev_py), (px, py), color=decay_val, thickness=1)
            else:
                img_np[1, py, px] = decay_val
        return torch.tensor(img_np, dtype=torch.float32)

    def __getitem__(self, idx):
        data = self.episodes[idx]
        
        cont = data['cont'].copy()
        target = data['target'].copy()
        
        # Augmentation Logic (Y-Flip)
        if self.augment and random.random() < 0.5:
            # 1. Flip Continuous Features
            cont[:, 1] = 1.0 - cont[:, 1] # start_y
            cont[:, 3] = 1.0 - cont[:, 3] # end_y_prev
            cont[:, 5] = -cont[:, 5]      # dy_prev (Vector Flip)
            target[1] = 1.0 - target[1]   # Target Flip
            
            # 2. Get Image
            if self.cache_images:
                # Retrieve Base Image from Cache
                base_img = self.img_cache[idx]
                # Apply Flip (Y-Flip = Flip H dimension = dim 1)
                img = torch.flip(base_img, [1])
            else:
                img = self._generate_image(cont) # Generate from flipped coords
        else:
            # No Augmentation
            if self.cache_images:
                img = self.img_cache[idx]
            else:
                img = self._generate_image(cont)
        
        cont_tensor = torch.tensor(cont, dtype=torch.float32)
        cat_tensor = torch.tensor(data['cat'], dtype=torch.long)
        target_tensor = torch.tensor(target, dtype=torch.float32)
        
        # Deep Supervision Targets
        seq_len = cont.shape[0]
        aux_target = np.zeros((seq_len, 2), dtype=np.float32)
        if seq_len > 1:
            aux_target[:-1] = cont[1:, 0:2]
        aux_target[-1] = target # Last step target is GT
        aux_target_tensor = torch.tensor(aux_target, dtype=torch.float32)
        
        return img, cont_tensor, cat_tensor, target_tensor, aux_target_tensor

## 3. 모델 아키텍처 (GRU 적용)

In [4]:
class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x_cat = torch.cat([avg_out, max_out], dim=1)
        x_out = self.conv(x_cat)
        return self.sigmoid(x_out)

class ImprovedCNN(nn.Module):
    def __init__(self):
        super(ImprovedCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(2, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(),
        )
        self.sa = SpatialAttention()
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(128, 128)
        
    def forward(self, x):
        x = self.features(x)
        sa_map = self.sa(x)
        x = x * sa_map
        x = self.pool(x).flatten(1)
        x = F.relu(self.fc(x))
        return x

class LSTMAttention(nn.Module):
    def __init__(self, hidden_dim):
        super(LSTMAttention, self).__init__()
        self.attention = nn.Linear(hidden_dim, 1)
    def forward(self, rnn_output):
        attn_weights = torch.softmax(self.attention(rnn_output), dim=1)
        return torch.sum(attn_weights * rnn_output, dim=1)

class MultiModalNetV6(nn.Module):
    def __init__(self, input_dim_cont, num_types, num_results, gru_hidden=96):
        super(MultiModalNetV6, self).__init__()
        self.cnn = ImprovedCNN()
        self.type_emb = nn.Embedding(num_types, 8)
        self.result_emb = nn.Embedding(num_results, 8)
        
        total_input_dim = input_dim_cont + 8 + 8
        self.gru_hidden = gru_hidden
        
        # [변경] Split GRU (Anti-Leakage)
        # Forward와 Backward를 물리적으로 분리하여, Forward GRU에 미래 정보가 유입되는 것을 원천 차단
        self.gru_fwd = nn.GRU(
            input_size=total_input_dim,
            hidden_size=gru_hidden,
            num_layers=2, # Multi-layer OK (Forward끼리만 연결됨)
            batch_first=True,
            bidirectional=False, # 단방향
            dropout=0.1
        )
        
        self.gru_bwd = nn.GRU(
            input_size=total_input_dim,
            hidden_size=gru_hidden,
            num_layers=2,
            batch_first=True,
            bidirectional=False, # 단방향
            dropout=0.1
        )
        
        self.gru_attn = LSTMAttention(gru_hidden * 2)
        self.gru_fc = nn.Linear(gru_hidden * 2, 128)
        
        # [New] Deep Supervision Head
        # Use ONLY Forward Hidden State for Aux Output
        self.aux_fc = nn.Linear(gru_hidden, 2) 
        
        self.fusion_fc = nn.Sequential(
            nn.BatchNorm1d(256), nn.Dropout(0.3),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 2)
        )

    def forward(self, img, cont, cat, lengths):
        # 1. Feature Extraction
        img_feat = self.cnn(img)
        emb_type = self.type_emb(cat[:, :, 0])
        emb_result = self.result_emb(cat[:, :, 1])
        x_seq = torch.cat([cont, emb_type, emb_result], dim=2)
        
        # 2. Forward GRU
        packed_fwd = pack_padded_sequence(x_seq, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out_fwd_packed, _ = self.gru_fwd(packed_fwd)
        out_fwd, _ = pad_packed_sequence(out_fwd_packed, batch_first=True)
        
        # 3. Backward GRU (Manual Flip)
        # Reverse sequence for backward pass
        # Note: Padding should remain at the end, so we need careful reversal?
        # pack_padded_sequence handles lengths correctly, verifying...
        # For simplicity and correctness with variable lengths:
        # We can just reverse the input tensor in time dim, pack, run GRU, unpack, reverse back.
        # But we need to handle padding.
        
        # Optimized Backward Strategy:
        # Since we use pack_padded_sequence with enforce_sorted=False, 
        # we can just flip the non-padded parts? Or just flip the whole tensor and let pack handle it?
        # Actually, PyTorch's pad_packed_sequence output is padded with zeros.
        # It's safer to rely on PyTorch's bidirectional implementation but we can't because of the leakage.
        # So manual flip is needed.
        
        # Manual Flip Logic:
        x_seq_bwd = x_seq.clone()
        for i, length in enumerate(lengths):
            x_seq_bwd[i, :length] = x_seq[i, :length].flip(0)
            
        packed_bwd = pack_padded_sequence(x_seq_bwd, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out_bwd_packed, _ = self.gru_bwd(packed_bwd)
        out_bwd, _ = pad_packed_sequence(out_bwd_packed, batch_first=True)
        
        # Reverse output back to original order
        for i, length in enumerate(lengths):
            out_bwd[i, :length] = out_bwd[i, :length].flip(0)
            
        # 4. Concatenate: [Forward, Backward]
        # out_fwd: [B, L, H], out_bwd: [B, L, H]
        gru_out_combined = torch.cat([out_fwd, out_bwd], dim=2) # [B, L, H*2]
        
        # 5. Main Output (Legacy)
        gru_ctx = self.gru_attn(gru_out_combined)
        seq_feat = F.relu(self.gru_fc(gru_ctx))
        concat_feat = torch.cat([img_feat, seq_feat], dim=1)
        final_out = self.fusion_fc(concat_feat)
        
        # 6. Aux Output (Deep Supervision)
        # Use ONLY out_fwd (Pure Past->Future info)
        aux_out = self.aux_fc(out_fwd) # [B, L, 2]
        
        return final_out, aux_out

class EuclideanLoss(nn.Module):
    def __init__(self):
        super(EuclideanLoss, self).__init__()
    def forward(self, pred, target):
        pred_real = pred * torch.tensor([105.0, 68.0], device=pred.device)
        target_real = target * torch.tensor([105.0, 68.0], device=target.device)
        return torch.mean(torch.sqrt(torch.sum((pred_real - target_real)**2, dim=1) + 1e-6))
        
class MaskedSeqEuclideanLoss(nn.Module):
    def __init__(self):
        super(MaskedSeqEuclideanLoss, self).__init__()
        
    def forward(self, pred, target, lengths):
        # pred: [B, L, 2]
        # target: [B, L, 2]
        # lengths: [B]
        
        mask = torch.arange(pred.size(1), device=pred.device)[None, :] < lengths[:, None]
        mask = mask.unsqueeze(-1) # [B, L, 1]
        
        pred_real = pred * torch.tensor([105.0, 68.0], device=pred.device)
        target_real = target * torch.tensor([105.0, 68.0], device=target.device)
        
        diff = pred_real - target_real
        dist = torch.sqrt(torch.sum(diff**2, dim=2) + 1e-6) # [B, L]
        
        # Masking
        dist = dist * mask.squeeze(-1)
        
        return dist.sum() / mask.sum()

## 4. 학습

In [5]:
def multimodal_collate_fn(batch):
    imgs, conts, cats, targets, aux_targets = zip(*batch)
    imgs_batched = torch.stack(imgs, dim=0)
    lengths = torch.tensor([len(c) for c in conts], dtype=torch.long)
    conts_padded = pad_sequence(conts, batch_first=True)
    cats_padded = pad_sequence(cats, batch_first=True)
    targets = torch.stack(targets, dim=0)
    aux_targets_padded = pad_sequence(aux_targets, batch_first=True)
    return imgs_batched, conts_padded, cats_padded, lengths, targets, aux_targets_padded

In [6]:
# [v7 Fix] Leakage-Free Training Function
# Now receives Raw DataFrame instead of transformed episodes
def train_multimodal_v8_team(train_df, n_splits=10, epochs=100, batch_size=64, lr=0.001):
    gkf = GroupKFold(n_splits=n_splits)
    groups = train_df['game_id'] 
    
    fold_scores = []

    # Get dimensions from first fold (will be same for all folds)
    input_dim_cont = None
    num_types = None
    num_results = None
    
    # K-Fold Loop
    for fold, (train_idx, val_idx) in enumerate(gkf.split(train_df, groups=groups)):
        
        # Set fold-specific seed for diversity
        fold_seed = FOLD_SEEDS[fold + 1]
        seed_everything(fold_seed)
        print(f"[Fold {fold+1}] Seed fixed to {fold_seed}")
        print(f"\n=== Fold {fold+1}/{n_splits} ===")
        
        # 1. Split Raw Data
        train_sub_df = train_df.iloc[train_idx].copy()
        val_sub_df = train_df.iloc[val_idx].copy()
        
        # 2. Fit Preprocessor (Inside Loop -> No Leakage)
        preprocessor = FootballPreprocessorMultimodalTeam()
        preprocessor.fit(train_sub_df)

        # Get dimensions (only once, on first fold)
        if input_dim_cont is None:
            input_dim_cont = preprocessor.get_input_dim()
            num_types, num_results, num_teams = preprocessor.get_num_classes()
            print(f"Model Dimensions: input={input_dim_cont}, types={num_types}, results={num_results}, teams={num_teams}")
        
        # 3. Transform Data
        # 3. Transform Data
        train_episodes = preprocessor.transform(train_sub_df, is_train=True)
        val_episodes = preprocessor.transform(val_sub_df, is_train=True) # Val also needs targets
        
        input_dim_cont = preprocessor.get_input_dim()
        num_types, num_results, num_teams = preprocessor.get_num_classes()
        
        # 4. Dataset & Loader (With Caching)
        # Train: Cache Enabled, Augment Enabled
        train_dataset = MultiModalDataset(train_episodes, augment=True, cache_images=True)
        # Val: Cache Enabled, No Augment
        val_dataset = MultiModalDataset(val_episodes, augment=False, cache_images=True)
        
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=multimodal_collate_fn, drop_last=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=multimodal_collate_fn)
        
        # Model (Split-GRU v6 Architecture)
        model = MultiModalNetV8Team(input_dim_cont, num_types, num_results, num_teams).to(DEVICE)
        
        criterion_main = EuclideanLoss()
        criterion_aux = MaskedSeqEuclideanLoss() 
        
        optimizer = optim.Adam(model.parameters(), lr=lr) 
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
        
        best_dist = float('inf')
        best_state = None
        patience_counter = 0
        patience_limit = 15
        
        for epoch in range(epochs):
            model.train()
            train_loss = 0
            
            # Tqdm for progress tracking (optional, maybe noisy for user logs)
            # for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            for batch in train_loader:
                imgs, cont, cat, lengths, target, aux_target = batch
                imgs, cont, cat, lengths, target, aux_target = imgs.to(DEVICE), cont.to(DEVICE), cat.to(DEVICE), lengths.to(DEVICE), target.to(DEVICE), aux_target.to(DEVICE)
                
                optimizer.zero_grad()
                pred, aux_pred = model(imgs, cont, cat, lengths)
                
                loss_m = criterion_main(pred, target)
                loss_a = criterion_aux(aux_pred, aux_target, lengths)
                
                loss = loss_m + 0.5 * loss_a
                loss.backward()
                optimizer.step()
                train_loss += loss.item()
            
            train_loss /= len(train_loader)
            
            model.eval()
            val_dists = []
            with torch.no_grad():
                for batch in val_loader:
                    imgs, cont, cat, lengths, target, aux_target = batch
                    imgs, cont, cat, lengths, target, aux_target = imgs.to(DEVICE), cont.to(DEVICE), cat.to(DEVICE), lengths.to(DEVICE), target.to(DEVICE), aux_target.to(DEVICE)
                    
                    pred, _ = model(imgs, cont, cat, lengths)
                    
                    pred_real = pred.cpu().numpy() * np.array([105.0, 68.0])
                    target_real = target.cpu().numpy() * np.array([105.0, 68.0])
                    val_dists.extend(np.sqrt(np.sum((pred_real - target_real)**2, axis=1)))
            
            mean_dist = np.mean(val_dists)
            scheduler.step(mean_dist)
            print(f"Epoch {epoch+1}: Train Loss {train_loss:.4f}, Val Dist {mean_dist:.4f}")
            
            if mean_dist < best_dist:
                best_dist = mean_dist
                best_state = model.state_dict()
                # Save with v7 suffix
                torch.save(best_state, f"models/multimodal_v8_team_h96_{fold+1}.pth")
                joblib.dump(preprocessor, f"models/preprocessor_v8_team_h96_{fold+1}.pkl")
                print(f"  -> Saved Best Model (Dist: {best_dist:.4f})")
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= patience_limit:
                    print("Early Stopping")
                    break
        
        print(f"Fold {fold+1} Finished. Best Valid Dist: {best_dist:.4f}")
        fold_scores.append(best_dist)
        
    print(f"Average Score: {np.mean(fold_scores):.4f}")

In [8]:
# [v7 Execution]
train_multimodal_v8_team(train_df, n_splits=10, epochs=100, batch_size=32, lr=0.001)

[Fold 1] Seed fixed to 42

=== Fold 1/10 ===
Model Dimensions: input=10, types=27, results=21, teams=13
Pre-rendering 13847 images (Cache Enabled)...


Caching Images: 100%|██████████| 13847/13847 [00:01<00:00, 13144.91it/s]


Pre-rendering 1581 images (Cache Enabled)...


Caching Images: 100%|██████████| 1581/1581 [00:00<00:00, 13372.69it/s]


Epoch 1: Train Loss 62.2199, Val Dist 20.9712
  -> Saved Best Model (Dist: 20.9712)
Epoch 2: Train Loss 49.2329, Val Dist 17.5269
  -> Saved Best Model (Dist: 17.5269)
Epoch 3: Train Loss 46.3847, Val Dist 18.1095
Epoch 4: Train Loss 44.8850, Val Dist 16.1248
  -> Saved Best Model (Dist: 16.1248)
Epoch 5: Train Loss 43.9164, Val Dist 15.2574
  -> Saved Best Model (Dist: 15.2574)
Epoch 6: Train Loss 43.2210, Val Dist 14.8752
  -> Saved Best Model (Dist: 14.8752)
Epoch 7: Train Loss 42.5923, Val Dist 14.7858
  -> Saved Best Model (Dist: 14.7858)
Epoch 8: Train Loss 42.1838, Val Dist 14.7978
Epoch 9: Train Loss 42.0644, Val Dist 14.9110
Epoch 10: Train Loss 41.3893, Val Dist 14.9113
Epoch 11: Train Loss 41.1888, Val Dist 14.3177
  -> Saved Best Model (Dist: 14.3177)
Epoch 12: Train Loss 40.8521, Val Dist 14.3704
Epoch 13: Train Loss 40.6391, Val Dist 14.7111
Epoch 14: Train Loss 40.4529, Val Dist 14.8804
Epoch 15: Train Loss 40.2241, Val Dist 14.5072
Epoch 16: Train Loss 39.7415, Val Dist

...

Epoch 1: Train Loss 62.3786, Val Dist 20.0246
  -> Saved Best Model (Dist: 20.0246)
Epoch 2: Train Loss 48.5787, Val Dist 18.2747
  -> Saved Best Model (Dist: 18.2747)
Epoch 3: Train Loss 45.9529, Val Dist 17.4680
  -> Saved Best Model (Dist: 17.4680)
Epoch 4: Train Loss 44.7546, Val Dist 16.4166
  -> Saved Best Model (Dist: 16.4166)
Epoch 5: Train Loss 43.8978, Val Dist 16.1957
  -> Saved Best Model (Dist: 16.1957)
Epoch 6: Train Loss 43.1706, Val Dist 15.9481
  -> Saved Best Model (Dist: 15.9481)
Epoch 7: Train Loss 42.5366, Val Dist 15.2530
  -> Saved Best Model (Dist: 15.2530)
Epoch 8: Train Loss 42.1998, Val Dist 15.3202
Epoch 9: Train Loss 41.8605, Val Dist 15.4219
Epoch 10: Train Loss 41.6160, Val Dist 15.4769
Epoch 11: Train Loss 41.2538, Val Dist 14.7449
  -> Saved Best Model (Dist: 14.7449)
Epoch 12: Train Loss 41.0054, Val Dist 14.8000
Epoch 13: Train Loss 40.9064, Val Dist 14.6936
  -> Saved Best Model (Dist: 14.6936)
Epoch 14: Train Loss 40.5695, Val Dist 14.5863
  -> Save

Caching Images: 100%|██████████| 13858/13858 [00:01<00:00, 12165.49it/s]


Pre-rendering 1570 images (Cache Enabled)...


Caching Images: 100%|██████████| 1570/1570 [00:00<00:00, 11533.41it/s]


Epoch 1: Train Loss 63.8255, Val Dist 20.0143
  -> Saved Best Model (Dist: 20.0143)
Epoch 2: Train Loss 48.9091, Val Dist 20.4999
Epoch 3: Train Loss 46.7099, Val Dist 16.5725
  -> Saved Best Model (Dist: 16.5725)
Epoch 4: Train Loss 44.9789, Val Dist 16.3499
  -> Saved Best Model (Dist: 16.3499)
Epoch 5: Train Loss 43.9715, Val Dist 15.5410
  -> Saved Best Model (Dist: 15.5410)
Epoch 6: Train Loss 43.1160, Val Dist 15.6383
Epoch 7: Train Loss 42.6044, Val Dist 15.5528
Epoch 8: Train Loss 42.2912, Val Dist 15.6191
Epoch 9: Train Loss 41.8790, Val Dist 14.9697
  -> Saved Best Model (Dist: 14.9697)
Epoch 10: Train Loss 41.5893, Val Dist 15.8647
Epoch 11: Train Loss 41.1334, Val Dist 14.4626
  -> Saved Best Model (Dist: 14.4626)
Epoch 12: Train Loss 40.9848, Val Dist 14.3685
  -> Saved Best Model (Dist: 14.3685)
Epoch 13: Train Loss 40.7242, Val Dist 14.7722
Epoch 14: Train Loss 40.4332, Val Dist 14.5370
Epoch 15: Train Loss 40.3339, Val Dist 14.4247
Epoch 16: Train Loss 40.1003, Val Dist

## 5. 추론 (Inference)

In [9]:
import os
import joblib
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import cv2
from tqdm import tqdm
from src.models_multimodal_v8_team import MultiModalNetV8Team

os.makedirs("submission_multimodal", exist_ok=True)

# =========================
# Test Dataset
# =========================
class TestMultiModalDataset(Dataset):
    def __init__(self, submission_df, preprocessor, img_size=(68, 105)):
        self.submission_df = submission_df
        self.preprocessor = preprocessor
        self.H, self.W = img_size

    def __len__(self):
        return len(self.submission_df)

    def _generate_image(self, cont_data):
        img_np = np.zeros((2, self.H, self.W), dtype=np.float32)
        if len(cont_data) == 0:
            return torch.tensor(img_np, dtype=torch.float32)

        seq_len = cont_data.shape[0]
        xs = cont_data[:, 0]
        ys = cont_data[:, 1]

        x_idxs = (xs * (self.W - 1)).round().astype(int).clip(0, self.W - 1)
        y_idxs = (ys * (self.H - 1)).round().astype(int).clip(0, self.H - 1)

        for t in range(seq_len):
            px, py = x_idxs[t], y_idxs[t]
            img_np[0, py, px] = 1.0
            decay_val = (t + 1) / seq_len
            if t > 0:
                prev_px, prev_py = x_idxs[t-1], y_idxs[t-1]
                cv2.line(img_np[1], (prev_px, prev_py), (px, py), color=decay_val, thickness=1)
            else:
                img_np[1, py, px] = decay_val

        return torch.tensor(img_np, dtype=torch.float32)

    def __getitem__(self, idx):
        row = self.submission_df.iloc[idx]
        raw_path = row['path']
        path = raw_path[2:] if raw_path.startswith("./") else raw_path

        df = pd.read_csv(path)
        episodes = self.preprocessor.transform(df, is_train=False)

        if len(episodes) == 0:
            input_dim = self.preprocessor.get_input_dim()
            cont = torch.zeros((1, input_dim), dtype=torch.float32)
            # Use Unknown indices (3 features: type, result, team)
            unk_type = self.preprocessor.type_encoder.transform(["Unknown"])[0]
            unk_result = self.preprocessor.result_encoder.transform(["Unknown"])[0]
            unk_team = self.preprocessor.team_encoder.transform(["Unknown"])[0]
            cat = torch.tensor([[unk_type, unk_result, unk_team]], dtype=torch.long)
            img = torch.zeros((2, self.H, self.W), dtype=torch.float32)
            return img, cont, cat

        ep = episodes[0]
        cont = torch.tensor(ep['cont'], dtype=torch.float32)
        cat = torch.tensor(ep['cat'], dtype=torch.long)
        img = self._generate_image(ep['cont'])

        return img, cont, cat

def test_mm_collate_fn(batch):
    imgs, conts, cats = zip(*batch)
    imgs_batched = torch.stack(imgs, dim=0)
    lengths = torch.tensor([len(c) for c in conts], dtype=torch.long)
    conts_padded = pad_sequence(conts, batch_first=True)
    cats_padded = pad_sequence(cats, batch_first=True)
    return imgs_batched, conts_padded, cats_padded, lengths

# =========================
# Load Submission Meta
# =========================
submission = pd.read_csv("sample_submission.csv")
test_meta = pd.read_csv("test.csv")
submission = submission.merge(test_meta, on="game_episode", how="left")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# =========================
# 10-Fold Inference with CORRECTED TTA
# =========================
NUM_FOLDS = 10
all_fold_preds = []

for fold in range(1, NUM_FOLDS + 1):
    print(f"\n=== Inference Fold {fold} with Corrected TTA ===")

    # 1. Load preprocessor
    preprocessor = joblib.load(f"models/preprocessor_v8_team_h96_{fold}.pkl")

    # 2. Dataset / Loader
    test_dataset = TestMultiModalDataset(submission, preprocessor)
    test_loader = DataLoader(
        test_dataset,
        batch_size=128,
        shuffle=False,
        collate_fn=test_mm_collate_fn
    )

    # 3. Load model
    input_dim_cont = preprocessor.get_input_dim()
    num_types, num_results, num_teams = preprocessor.get_num_classes()
    print(f"Model Dimensions: input={input_dim_cont}, types={num_types}, results={num_results}, teams={num_teams}")

    model = MultiModalNetV8Team(input_dim_cont, num_types, num_results, num_teams).to(DEVICE)
    model.load_state_dict(torch.load(f"models/multimodal_v8_team_h96_{fold}.pth", map_location=DEVICE))
    model.eval()

    fold_preds = []

    # 4. Inference with CORRECTED TTA
    with torch.no_grad():
        for imgs, cont, cat, lengths in tqdm(test_loader, desc=f"Fold {fold}"):
            imgs, cont, cat, lengths = (
                imgs.to(DEVICE),
                cont.to(DEVICE),
                cat.to(DEVICE),
                lengths.to(DEVICE),
            )

            # -------------------------
            # (1) 원본 예측
            # -------------------------
            pred1, _ = model(imgs, cont, cat, lengths)

            # -------------------------
            # (2) Y-Flip TTA 예측 (CORRECTED!)
            # -------------------------
            imgs_f = torch.flip(imgs, dims=[2])  # Y-axis flip

            # ✅ CORRECTED: Raw → Flip → Scale
            scaler = preprocessor.scaler
            mean = torch.tensor(scaler.mean_, device=cont.device, dtype=cont.dtype)
            std = torch.tensor(scaler.scale_, device=cont.device, dtype=cont.dtype)

            # Step 1: z-score → raw(norm)
            cont_raw = cont * std + mean

            # Step 2: Y-Flip in raw space
            cont_raw[:, :, 1] = 1.0 - cont_raw[:, :, 1]  # start_y_norm
            cont_raw[:, :, 3] = 1.0 - cont_raw[:, :, 3]  # end_y_prev_norm
            cont_raw[:, :, 5] = -cont_raw[:, :, 5]       # dy_prev_norm

            # Step 3: raw → z-score
            cont_f = (cont_raw - mean) / std

            pred2, _ = model(imgs_f, cont_f, cat, lengths)

            # 예측값 다시 flip
            pred2[:, 1] = 1.0 - pred2[:, 1]

            # -------------------------
            # (3) 평균
            # -------------------------
            pred = 0.5 * (pred1 + pred2)

            pred_np = pred.cpu().numpy()
            pred_np[:, 0] *= 105.0
            pred_np[:, 1] *= 68.0
            fold_preds.append(pred_np)

    fold_preds = np.vstack(fold_preds)
    all_fold_preds.append(fold_preds)

# =========================
# Ensemble (Mean)
# =========================
final_preds = np.mean(all_fold_preds, axis=0)

submission["end_x"] = final_preds[:, 0].clip(0, 105)
submission["end_y"] = final_preds[:, 1].clip(0, 68)

submission[["game_episode", "end_x", "end_y"]].to_csv(
    "submission_multimodal/multimodal_v8_team_h96_tta_corrected.csv", index=False
)

print("\n✅ Saved: submission_multimodal/multimodal_v8_team_h96_tta_corrected.csv")
print("📊 TTA Fix Applied: Raw → Flip → Scale")


C:\Users\semic\AppData\Local\Temp\ipykernel_102520\1005457692.py:121: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f"models/multimodal_v8_t

Device: cuda

=== Inference Fold 1 with Corrected TTA ===
Model Dimensions: input=10, types=27, results=21, teams=13


Fold 1: 100%|██████████| 19/19 [01:20<00:00,  4.22s/it]
C:\Users\semic\AppData\Local\Temp\ipykernel_102520\1005457692.py:121: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  m


=== Inference Fold 2 with Corrected TTA ===
Model Dimensions: input=10, types=27, results=21, teams=13


Fold 2: 100%|██████████| 19/19 [00:53<00:00,  2.83s/it]
C:\Users\semic\AppData\Local\Temp\ipykernel_102520\1005457692.py:121: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  m

...


=== Inference Fold 9 with Corrected TTA ===
Model Dimensions: input=10, types=26, results=20, teams=13


Fold 9: 100%|██████████| 19/19 [00:52<00:00,  2.76s/it]
C:\Users\semic\AppData\Local\Temp\ipykernel_102520\1005457692.py:121: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  m


=== Inference Fold 10 with Corrected TTA ===
Model Dimensions: input=10, types=27, results=21, teams=13


Fold 10: 100%|██████████| 19/19 [00:51<00:00,  2.69s/it]


✅ Saved: submission_multimodal/multimodal_v8_team_h96_tta_corrected.csv
📊 TTA Fix Applied: Raw → Flip → Scale


In [12]:
import pandas as pd
import os

os.makedirs("submission_ensemble", exist_ok=True)

h96 = pd.read_csv("submission_multimodal/multimodal_v7_h96_tta_corrected.csv")
polar = pd.read_csv("submission_polar_h96_optimized.csv")
team_h96 = pd.read_csv("submission_multimodal/multimodal_v8_team_h96_tta_corrected.csv")

# map (순서 유지)
polar_map = polar.set_index("game_episode")[["end_x","end_y"]]
team_map = team_h96.set_index("game_episode")[["end_x","end_y"]]

px = h96["game_episode"].map(polar_map["end_x"])
py = h96["game_episode"].map(polar_map["end_y"])
tx = h96["game_episode"].map(team_map["end_x"])
ty = h96["game_episode"].map(team_map["end_y"])

# weights
w_team = 0.05  # 0.10도 한 번만 실험
w_h96 = 0.6 * (1 - w_team)
w_polar = 0.4 * (1 - w_team)

out = h96.copy()
out["end_x"] = w_h96*h96["end_x"] + w_polar*px + w_team*tx
out["end_y"] = w_h96*h96["end_y"] + w_polar*py + w_team*ty

out["end_x"] = out["end_x"].clip(0, 105.0)
out["end_y"] = out["end_y"].clip(0, 68.0)

path = "submission_ensemble/ens_h96_polar_teamh96_05.csv"
out.to_csv(path, index=False)
print("saved:", path)

display(out.head())

saved: submission_ensemble/ens_h96_polar_teamh96_05.csv


,game_episode,end_x,end_y
0,153363_1,65.353189,11.358270
1,153363_2,27.275434,50.600202
2,153363_6,30.289655,62.947516
3,153363_7,52.675758,6.015957
4,153363_8,81.028175,8.725982


In [13]:
import pandas as pd
import os

os.makedirs("submission_ensemble", exist_ok=True)

h96 = pd.read_csv("submission_multimodal/multimodal_v7_h96_tta_corrected.csv")
polar = pd.read_csv("submission_polar_h96_optimized.csv")
team_h96 = pd.read_csv("submission_multimodal/multimodal_v8_team_h96_tta_corrected.csv")

# map (순서 유지)
polar_map = polar.set_index("game_episode")[["end_x","end_y"]]
team_map = team_h96.set_index("game_episode")[["end_x","end_y"]]

px = h96["game_episode"].map(polar_map["end_x"])
py = h96["game_episode"].map(polar_map["end_y"])
tx = h96["game_episode"].map(team_map["end_x"])
ty = h96["game_episode"].map(team_map["end_y"])

# weights
w_team = 0.1  # 0.10도 한 번만 실험
w_h96 = 0.6 * (1 - w_team)
w_polar = 0.4 * (1 - w_team)

out = h96.copy()
out["end_x"] = w_h96*h96["end_x"] + w_polar*px + w_team*tx
out["end_y"] = w_h96*h96["end_y"] + w_polar*py + w_team*ty

out["end_x"] = out["end_x"].clip(0, 105.0)
out["end_y"] = out["end_y"].clip(0, 68.0)

path = "submission_ensemble/ens_h96_polar_teamh96_10.csv"
out.to_csv(path, index=False)
print("saved:", path)

display(out.head())

saved: submission_ensemble/ens_h96_polar_teamh96_10.csv


,game_episode,end_x,end_y
0,153363_1,65.277999,11.306762
1,153363_2,27.239527,50.755751
2,153363_6,30.347696,63.039867
3,153363_7,52.773866,5.919607
4,153363_8,81.017477,8.717041


In [15]:
result_v7_foldseed_h96_tta_corrected = pd.read_csv("submission_multimodal/multimodal_v7_h96_tta_corrected.csv")
display(result_v7_foldseed_h96_tta_corrected.head())

result_v8_team_h96_tta_corrected = pd.read_csv("submission_multimodal/multimodal_v8_team_h96_tta_corrected.csv")
display(result_v8_team_h96_tta_corrected.head())

result_v7_h96_polar = pd.read_csv("submission_polar_h96_optimized.csv")
display(result_v7_h96_polar.head())

result_v7_h96_polar = pd.read_csv("submission_ensemble/ens_h9660_polar40.csv")
display(result_v7_h96_polar.head())

result_ens_h96_polar_team05 = pd.read_csv("submission_ensemble/ens_h96_polar_teamh96_05.csv")
display(result_ens_h96_polar_team05.head())

result_ens_h96_polar_team10 = pd.read_csv("submission_ensemble/ens_h96_polar_teamh96_10.csv")
display(result_ens_h96_polar_team10.head())

,game_episode,end_x,end_y
0,153363_1,63.744760,10.535255
1,153363_2,25.247570,50.701256
2,153363_6,26.461975,63.656200
3,153363_7,53.256310,4.663423
4,153363_8,81.452260,8.473086


,game_episode,end_x,end_y
0,153363_1,63.924590,10.379606
1,153363_2,26.593204,53.555634
2,153363_6,31.392430,64.702194
3,153363_7,54.539806,4.185305
4,153363_8,80.824910,8.556105


,game_episode,end_x,end_y
0,153363_1,67.953806,12.721565
1,153363_2,30.406996,50.059748
2,153363_6,35.886074,61.653611
3,153363_7,51.559660,8.285633
4,153363_8,80.418792,9.127678


,game_episode,end_x,end_y
0,153363_1,65.428378,11.409779
1,153363_2,27.311340,50.444653
2,153363_6,30.231615,62.855164
3,153363_7,52.577650,6.112307
4,153363_8,81.038873,8.734923


,game_episode,end_x,end_y
0,153363_1,65.353189,11.358270
1,153363_2,27.275434,50.600202
2,153363_6,30.289655,62.947516
3,153363_7,52.675758,6.015957
4,153363_8,81.028175,8.725982


,game_episode,end_x,end_y
0,153363_1,65.277999,11.306762
1,153363_2,27.239527,50.755751
2,153363_6,30.347696,63.039867
3,153363_7,52.773866,5.919607
4,153363_8,81.017477,8.717041


In [22]:
result_v7_foldseed_h96_tta_corrected = pd.read_csv("submission_multimodal/multimodal_v7_h96_tta_corrected.csv")
display(result_v7_foldseed_h96_tta_corrected.head(10))

result_Final_env_v2_3 = pd.read_csv("submission_multimodal/Final_env_v2_4.csv")
display(result_Final_env_v2_3.head(10))

result_v7_h96_polar = pd.read_csv("submission_polar_h96_optimized.csv")
display(result_v7_h96_polar.head(10))

result_v7_h96_polar = pd.read_csv("submission_ensemble/ens_h9660_polar40.csv")
display(result_v7_h96_polar.head(10))

,game_episode,end_x,end_y
0,153363_1,63.744760,10.535255
1,153363_2,25.247570,50.701256
2,153363_6,26.461975,63.656200
3,153363_7,53.256310,4.663423
4,153363_8,81.452260,8.473086
5,153363_9,73.318080,66.097336
6,153363_10,65.394610,6.111311
7,153363_12,70.400880,5.978782
8,153363_13,31.906475,64.938890
9,153363_15,70.288500,9.383939


,game_episode,end_x,end_y
0,153363_1,63.371765,12.353533
1,153363_2,25.322931,44.783375
2,153363_6,32.662900,63.261932
3,153363_7,55.991160,5.556868
4,153363_8,80.957960,8.612200
5,153363_9,72.582650,66.721050
6,153363_10,65.979020,7.507459
7,153363_12,70.334206,6.224345
8,153363_13,32.420254,65.724540
9,153363_15,70.136340,11.219527


,game_episode,end_x,end_y
0,153363_1,67.953806,12.721565
1,153363_2,30.406996,50.059748
2,153363_6,35.886074,61.653611
3,153363_7,51.559660,8.285633
4,153363_8,80.418792,9.127678
5,153363_9,76.223418,66.012002
6,153363_10,64.185196,7.951764
7,153363_12,76.645691,8.798978
8,153363_13,32.859031,63.211951
9,153363_15,74.895893,13.939054


,game_episode,end_x,end_y
0,153363_1,65.428378,11.409779
1,153363_2,27.311340,50.444653
2,153363_6,30.231615,62.855164
3,153363_7,52.577650,6.112307
4,153363_8,81.038873,8.734923
5,153363_9,74.480215,66.063202
6,153363_10,64.910844,6.847492
7,153363_12,72.898804,7.106860
8,153363_13,32.287497,64.248114
9,153363_15,72.131457,11.205985


In [7]:
import os
import pandas as pd

# =========================
# output folder
# =========================
os.makedirs("submission_ensemble", exist_ok=True)

# =========================
# load
# =========================
h96 = pd.read_csv("submission_multimodal/multimodal_v7_h96_tta_corrected.csv")
polar = pd.read_csv("submission_polar_h96_optimized.csv")
env = pd.read_csv("submission_multimodal/Final_env_v2_4_last.csv")

# =========================
# mapping (순서 유지)
# =========================
polar_map = polar.set_index("game_episode")[["end_x", "end_y"]]
env_map = env.set_index("game_episode")[["end_x", "end_y"]]

px = h96["game_episode"].map(polar_map["end_x"])
py = h96["game_episode"].map(polar_map["end_y"])
ex = h96["game_episode"].map(env_map["end_x"])
ey = h96["game_episode"].map(env_map["end_y"])

FIELD_X, FIELD_Y = 105.0, 68.0

# =========================
# ensemble helper
# =========================
def make_submission(name, w_h96, w_polar, w_env):
    out = h96.copy()
    out["end_x"] = w_h96 * h96["end_x"] + w_polar * px + w_env * ex
    out["end_y"] = w_h96 * h96["end_y"] + w_polar * py + w_env * ey

    out["end_x"] = out["end_x"].clip(0, FIELD_X)
    out["end_y"] = out["end_y"].clip(0, FIELD_Y)

    path = f"submission_ensemble/{name}.csv"
    out.to_csv(path, index=False)
    print(f"saved: {path} | weights = (h96={w_h96}, polar={w_polar}, env={w_env})")

# =========================
# generate submissions
# =========================
# make_submission("ens_h96_60_polar_30_env_10", 0.6, 0.3, 0.1)
# make_submission("ens_h96_60_polar_20_env_20", 0.6, 0.2, 0.2)
# make_submission("ens_h96_60_polar_10_env_30", 0.6, 0.1, 0.3)
# make_submission("ens_h96_40_polar_30_env_30", 0.4, 0.3, 0.3)
# make_submission("ens_h96_70_polar_20_env_10", 0.7, 0.2, 0.1)
# make_submission("ensemble_last", 0.35, 0.3, 0.35)
make_submission("ensemble_final", 0.5, 0.0, 0.5)


# 참고용 (올인)
# make_submission("ens_h96_60_env_40", 0.6, 0.0, 0.4)


saved: submission_ensemble/ensemble_final.csv | weights = (h96=0.5, polar=0.0, env=0.5)


In [8]:
import pandas as pd

files = [
    "ens_h96_60_polar_30_env_10.csv",
    "ensemble_last.csv",
    "ensemble_final.csv",
    "ens_h96_70_polar_20_env_10.csv",
    "ens_h96_60_polar_20_env_20.csv",
    "ens_h96_60_polar_10_env_30.csv",
    "ens_h96_40_polar_30_env_30.csv",
    
    "ens_h96_60_env_40.csv",
]

for f in files:
    print(f"\n=== {f} ===")
    display(pd.read_csv(f"submission_ensemble/{f}").head(10))



=== ens_h96_60_polar_30_env_10.csv ===


,game_episode,end_x,end_y
0,153363_1,64.970174,11.372976
1,153363_2,26.802934,49.917016
2,153363_6,29.909297,63.015996
3,153363_7,53.020800,5.839430
4,153363_8,81.092790,8.683375
5,153363_9,74.116138,66.134107
6,153363_10,65.090227,6.803061
7,153363_12,72.267656,6.849397
8,153363_13,32.243620,64.499373
9,153363_15,71.655502,10.934032



=== ensemble_last.csv ===


,game_episode,end_x,end_y
0,153363_1,64.449568,11.824238
1,153363_2,26.232012,49.900132
2,153363_6,31.087546,62.984044
3,153363_7,54.808754,6.294260
4,153363_8,80.698590,8.555894
5,153363_9,73.995906,66.458638
6,153363_10,66.273285,7.458243
7,153363_12,72.119154,6.584272
8,153363_13,32.897923,64.489807
9,153363_15,71.962726,11.100782



=== ensemble_final.csv ===


...

,game_episode,end_x,end_y
0,153363_1,64.053766,11.299369
1,153363_2,25.786121,48.861741
2,153363_6,29.264662,63.337661
3,153363_7,53.907100,5.293677
4,153363_8,81.200623,8.580279
5,153363_9,73.387985,66.275917
6,153363_10,65.448992,6.714200
7,153363_12,71.005359,6.334470
8,153363_13,32.155864,65.001891
9,153363_15,70.703591,10.390127



=== ens_h96_40_polar_30_env_30.csv ===


,game_episode,end_x,end_y
0,153363_1,64.895575,11.736631
1,153363_2,26.818006,48.733439
2,153363_6,31.149482,62.937143
3,153363_7,53.567770,6.018119
4,153363_8,80.993930,8.711198
5,153363_9,73.969052,66.258850
6,153363_10,65.207109,7.082291
7,153363_12,72.254321,6.898510
8,153363_13,32.346376,64.656503
9,153363_15,71.625070,11.301150



=== ens_h96_60_env_40.csv ===


,game_episode,end_x,end_y
0,153363_1,63.595562,11.262566
1,153363_2,25.277714,48.334104
2,153363_6,28.942345,63.498493
3,153363_7,54.350250,5.020801
4,153363_8,81.254540,8.528732
5,153363_9,73.023908,66.346822
6,153363_10,65.628374,6.669770
7,153363_12,70.374210,6.077007
8,153363_13,32.111987,65.253150
9,153363_15,70.227636,10.118174
